This notebook allows us to observe the state of the NVD at different years in the past. Here we scrape the NVD CVE Change History API for all changes related to CVE-CWE mappings and sort them by date and undo the changes year by year to get the state of the NVD at any year in the past.

In [1]:
import requests
from datetime import datetime, timedelta
import time
import os
import json
import re
import pandas as pd

We load all the CVE data from the NVD website. They are JSON formmatted.

In [4]:
folder_path = "../FixV2W/csv" 
all_cve_data = []

# Loop through all years 2002 → 2025
for year in range(2002, 2026):
    filename = f"nvdcve-2.0-{year}.json"
    file_path = os.path.join(folder_path, filename)
    print(year)
    
    if os.path.exists(file_path):
        with open(file_path, "r",  encoding="utf-8") as f:
            try:
                data = json.load(f)
                all_cve_data.extend(data['vulnerabilities'])
            except:
                continue

2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025


In [5]:
cwe_disc = ["CWE-119", "CWE-20", "CWE-200", "CWE-269", "CWE-287",
            "CWE-311", "CWE-330", "CWE-345", "CWE-400", "CWE-610",
            "CWE-662", "CWE-665", "CWE-668", "CWE-682", "CWE-697",
            "CWE-74", "CWE-755", "CWE-834", "CWE-284", "CWE-285",
           "CWE-693", "CWE-788"]
df_cwe_cate = pd.read_csv("../preprocess/csv_file/2021aug/cwe_category.csv")
df_cwe_view = pd.read_csv("../preprocess/csv_file/2021aug/cwe_view.csv")
cwe_proh = df_cwe_cate["ID"].tolist() + df_cwe_view["ID"].tolist() + ['CWE-17']

In [6]:
cve2cwe_list = []
for cve_data in all_cve_data:
    cve = cve_data['cve']['id']
    try:
        weaknesses = cve_data['cve']['weaknesses']
        cwes = []
        for cwe in weaknesses:
            cwes.append(cwe['description'][0]['value'])
    except KeyError:
        cwes = []
    #desc = cve_data['cve']['descriptions'][0]['value']
    cve2cwe_list.append([cve, cwes])

The following code calculates the totaly current number of discouraged and prohibited CWE mappings. "Current" refers to the time when all the JSON data was downloaded from the NVD website.

In [11]:
proh = 0
disc = 0
total = 0
for mapping in cve2cwe_list:
    for w in mapping[1]:
        if w in cwe_proh:
            proh += 1
        if w in cwe_disc:
            disc += 1
    total += 1

In [13]:
print(total, disc, proh)

313518 57214 13854


In [15]:
df = pd.DataFrame(cve2cwe_list, columns=['cve', 'cwe'])

In [17]:
df.to_csv("../long_analysis_data/current_mappings.csv", index = False)

In [19]:
changes = pd.read_csv("../long_analysis_data/combined_remaps.csv")

The following cell reverts the changes by one year (to 2025).

In [23]:
df = changes
df_2025 = df[df['date'].str.startswith('2025')].copy()

# Convert cve2cwe_list to dict for easier modification
cve2cwe_dict = {cve: set(cwes) for cve, cwes in cve2cwe_list}

# Track changes
revert_log = []

for _, row in df_2025.iterrows():
    cve = row['cve']
    old_cwe = row['old_cwe']
    new_cwe = row['new_cwe']
    
    if cve not in cve2cwe_dict:
        continue
    
    cwes = cve2cwe_dict[cve]
    before = cwes.copy()
    
    # Case 1: Removed mapping (new='*', old is real CWE) → add old CWE back
    if new_cwe == '*' and old_cwe != '*':
        cwes.add(old_cwe)
        revert_log.append([cve, 'added', old_cwe])
    
    # Case 2: Added mapping (old='*', new is real CWE) → remove new CWE
    elif old_cwe == '*' and new_cwe != '*':
        if new_cwe in cwes:
            cwes.remove(new_cwe)
            revert_log.append([cve, 'removed', new_cwe])
    
    # Case 3: Replaced (both real CWEs)
    elif old_cwe != '*' and new_cwe != '*':
        if new_cwe in cwes:
            cwes.remove(new_cwe)
        cwes.add(old_cwe)
        revert_log.append([cve, new_cwe, old_cwe])
    
    after = cwes
    if before != after:
        cve2cwe_dict[cve] = after

cve2cwe_list_2024 = [[cve, sorted(list(cwes))] for cve, cwes in cve2cwe_dict.items()]
reverted_cve2cwe = [[cve, sorted(list(cwes))] for cve, cwes in cve2cwe_dict.items()]


In [25]:
counts_imp = [[2025, total, disc, proh]]
for year in range(2024, 1998, -1):
    # we only want to include cve until the year we are analyzing, so filter to only those 
    cve_not = [f'CVE-{y}' for y in range(year+1, 2026)]
    total_count = 0
    proh = 0
    disc = 0
    for entry in reverted_cve2cwe:
        if not entry[0].startswith(tuple(cve_not)):
            total_count += 1
            for cwe in entry[1]:
                if cwe in cwe_proh:
                    proh += 1
                if cwe in cwe_disc:
                    disc += 1
    
    df = changes
    df_2025 = df[df['date'].str.startswith(f'{year+1}')].copy()
    
    # Convert cve2cwe_list to dict for easier modification
    cve2cwe_dict = {cve: set(cwes) for cve, cwes in reverted_cve2cwe}
    
    # Track 
    revert_log = []
    
    for _, row in df_2025.iterrows():
        cve = row['cve']
        old_cwe = row['old_cwe']
        new_cwe = row['new_cwe']
        
        if cve not in cve2cwe_dict:
            continue
        
        cwes = cve2cwe_dict[cve]
        before = cwes.copy()
        
        # Case 1: Removed mapping (new='*', old is real CWE) → add old CWE back
        if new_cwe == '*' and old_cwe != '*':
            cwes.add(old_cwe)
            revert_log.append(f"{cve}: added back {old_cwe}")
        
        # Case 2: Added mapping (old='*', new is real CWE) → remove new CWE
        elif old_cwe == '*' and new_cwe != '*':
            if new_cwe in cwes:
                cwes.remove(new_cwe)
                revert_log.append(f"{cve}: removed {new_cwe}")
        
        # Case 3: Replaced (both real CWEs)
        elif old_cwe != '*' and new_cwe != '*':
            if new_cwe in cwes:
                cwes.remove(new_cwe)
            cwes.add(old_cwe)
            revert_log.append(f"{cve}: replaced {new_cwe} → {old_cwe}")
        
        after = cwes
        if before != after:
            cve2cwe_dict[cve] = after
    
    # Convert back to list form
    reverted_cve2cwe = [[cve, sorted(list(cwes))] for cve, cwes in cve2cwe_dict.items()]    
    print(f"{year} & {total_count} & {disc} & {disc/float(total_count):.3f} & {proh} & {proh/float(total_count):.3f}\\\\")
    counts_imp.append([year, total_count, disc, proh])

2024 & 283210 & 46332 & 0.164 & 13764 & 0.049\\
2023 & 244513 & 42565 & 0.174 & 13695 & 0.056\\
2022 & 214717 & 38906 & 0.181 & 13811 & 0.064\\
2021 & 188006 & 36529 & 0.194 & 13762 & 0.073\\
2020 & 164933 & 33412 & 0.203 & 13862 & 0.084\\
2019 & 144332 & 31466 & 0.218 & 13894 & 0.096\\
2018 & 127301 & 29626 & 0.233 & 14381 & 0.113\\
2017 & 109809 & 25470 & 0.232 & 15866 & 0.144\\
2016 & 92788 & 19082 & 0.206 & 13837 & 0.149\\
2015 & 82232 & 15238 & 0.185 & 12197 & 0.148\\
2014 & 73478 & 12334 & 0.168 & 10545 & 0.144\\
2013 & 64493 & 10276 & 0.159 & 7700 & 0.119\\
2012 & 57711 & 8411 & 0.146 & 6343 & 0.110\\
2011 & 51818 & 6935 & 0.134 & 5145 & 0.099\\
2010 & 46956 & 5321 & 0.113 & 4084 & 0.087\\
2009 & 41739 & 4194 & 0.100 & 3164 & 0.076\\
2008 & 36699 & 3069 & 0.084 & 2236 & 0.061\\
2007 & 29523 & 1571 & 0.053 & 1001 & 0.034\\
2006 & 22943 & 696 & 0.030 & 481 & 0.021\\
2005 & 15800 & 434 & 0.027 & 290 & 0.018\\
2004 & 11032 & 318 & 0.029 & 202 & 0.018\\
2003 & 8325 & 265 & 0.032 & 15

In [26]:
for c in counts_imp:
    print(f'({c[0]},{c[3]})')

(2025,13854)
(2024,13764)
(2023,13695)
(2022,13811)
(2021,13762)
(2020,13862)
(2019,13894)
(2018,14381)
(2017,15866)
(2016,13837)
(2015,12197)
(2014,10545)
(2013,7700)
(2012,6343)
(2011,5145)
(2010,4084)
(2009,3164)
(2008,2236)
(2007,1001)
(2006,481)
(2005,290)
(2004,202)
(2003,158)
(2002,94)
(2001,31)
(2000,23)
(1999,19)


In [27]:
for c in counts_imp:
    print(f'({c[0]},{c[2]})')

(2025,57214)
(2024,46332)
(2023,42565)
(2022,38906)
(2021,36529)
(2020,33412)
(2019,31466)
(2018,29626)
(2017,25470)
(2016,19082)
(2015,15238)
(2014,12334)
(2013,10276)
(2012,8411)
(2011,6935)
(2010,5321)
(2009,4194)
(2008,3069)
(2007,1571)
(2006,696)
(2005,434)
(2004,318)
(2003,265)
(2002,139)
(2001,40)
(2000,28)
(1999,22)
